In [ ]:
# In[1]:

import pandas as pd
import pytz

# load metric data (cached as df_metric for kernel statefulness)
df_metric = pd.read_csv("metric.csv")

# ensure value is numeric and drop rows with NaN values in value
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_metric = df_metric.dropna(subset=['value'])

# convert timestamp to timezone-aware datetime in UTC+8 (Asia/Shanghai)
tz = pytz.timezone('Asia/Shanghai')
df_metric['timestamp_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# aggregate statistics per (cmdb_id, kpi_name) over the full series
agg = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(
        count='count',
        min='min',
        max='max',
        mean='mean',
        p50=lambda x: x.quantile(0.50),
        p90=lambda x: x.quantile(0.90),
        p95=lambda x: x.quantile(0.95),
    )
    .reset_index()
)

# round numeric summaries for compact display
num_cols = ['min', 'max', 'mean', 'p50', 'p90', 'p95']
agg[num_cols] = agg[num_cols].round(6)

# top 50 rows sorted by count descending
agg_top50 = agg.sort_values('count', ascending=False).head(50)

# full lists of distinct cmdb_id and kpi_name with their counts
cmdb_counts = df_metric['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count')
kpi_counts = df_metric['kpi_name'].value_counts().rename_axis('kpi_name').reset_index(name='count')

# final compact outputs: (1) aggregated table top 50, (2) cmdb counts, (3) kpi counts
agg_top50, cmdb_counts, kpi_counts

```
Out[1]:
```
```python
# Summarize results in plain English using existing kernel variables:
# df_metric, agg_top50, cmdb_counts, kpi_counts are expected to be present from prior run.

top_cmdb = ", ".join(f"{r.cmdb_id}({int(r.count)})" for r in cmdb_counts.head(5).itertuples())
top_kpis = ", ".join(f"{r.kpi_name}({int(r.count)})" for r in kpi_counts.head(5).itertuples())

distinct_cmdb_count = cmdb_counts.shape[0]
distinct_kpi_count = kpi_counts.shape[0]

top50_max_count = int(agg_top50['count'].max())
top50_median_count = float(agg_top50['count'].median())

summary = (
    f"Summary of metric.csv aggregation:\n"
    f"- Distinct cmdb_id values: {distinct_cmdb_count}\n"
    f"- Distinct kpi_name values: {distinct_kpi_count}\n"
    f"- Top 5 cmdb_id by sample count: {top_cmdb}\n"
    f"- Top 5 kpi_name by sample count: {top_kpis}\n"
    f"- Aggregation performed per (cmdb_id, kpi_name) producing: count, min, max, mean, median(p50), p90, p95.\n"
    f"- Returned table: top 50 (cmdb_id, kpi_name) pairs sorted by count. In that top-50 set, the maximum per-pair count = {top50_max_count}, "
    f"median per-pair count = {top50_median_count}.\n"
    f"- Note: The displayed top-50 table (agg_top50) contains compact numeric summaries for each pair (no raw time series)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  count           min           max          mean           p50           p90           p95
1580  apache01   OSLinux-OSLinux_PROCESS_apache_10001_PROCPPCount     31  9.000000e+00  1.000000e+01  9.225806e+00  9.000000e+00  1.000000e+01  1.000000e+01
12        IG01                            OSLinux-CPU_CPU_CPULoad     31  1.000000e+00  1.200000e+00  1.035161e+00  1.010000e+00  1.100000e+00  1.155000e+00
1668  apache02                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31  6.734629e+06  6.736436e+06  6.735531e+06  6.735531e+06  6.736254e+06  6.736344e+06
0         IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax     31  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09
1577  apache01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
...        ...                                                ...    ...           ...           ...           ...           ...           ...           ...
1506  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  1.120000e+02  1.120000e+02  1.120000e+02  1.120000e+02  1.120000e+02  1.120000e+02
102       IG02                        OSLinux-CPU_CPU_CPUUserTime     31  2.531900e+01  2.561224e+01  2.541199e+01  2.540943e+01  2.545720e+01  2.550495e+01
103       IG02                             OSLinux-CPU_CPU_CPUWio     31  2.078300e-02  2.343060e+01  4.611112e+00  3.117980e-01  2.321760e+01  2.334079e+01
104       IG02                        OSLinux-CPU_CPU_CPUidleutil     31  4.880330e+01  7.391210e+01  6.899343e+01  7.349297e+01  7.382991e+01  7.384745e+01
1498  Tomcat04     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00

[50 rows x 9 columns],          cmdb_id  count
0        Mysql02   3852
1        Mysql01   3811
2        Redis02   2204
3        Redis01   2201
4       Tomcat03   1583
..           ...    ...
24  ServiceTest7    124
25      dockerB1     33
26      dockerA2     28
27      dockerB2     11
28      dockerA1     10

[29 rows x 2 columns],                                               kpi_name  count
0                   OSLinux-OSLinux_ZABBIX_Host_Uptime    364
1           OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount    364
2         OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    364
3        OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies    364
4       OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    360
..                                                 ...    ...
342  Container-DOCKER_CONTAINER_d27123361435--bcou-...      2
343          OSLinux-OSLinux_MEMORY_MEMORY_MEMTotalMem      2
344  Container-DOCKER_CONTAINER_23bdcf67c7e3--bcou-...      1
345  Container-DOCKER_CONTAINER_23bdcf67c7e3--bcou-...      1
346  Container-DOCKER_CONTAINER_d27123361435--bcou-...      1

[347 rows x 2 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# reuse df_metric if present, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

# ensure numeric and drop NaNs in value
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_metric = df_metric.dropna(subset=['value'])

# ensure timezone-aware timestamp in UTC+8 (Asia/Shanghai)
tz = pytz.timezone('Asia/Shanghai')
if 'timestamp_dt' not in df_metric.columns:
    df_metric['timestamp_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure dtype is datetime with tz
    if df_metric['timestamp_dt'].dt.tz is None:
        df_metric['timestamp_dt'] = pd.to_datetime(df_metric['timestamp_dt']).dt.tz_localize(tz)

# 1) Compute global p95 and p5 per (cmdb_id, kpi_name) using the full series
global_thresh = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(global_p95=lambda x: x.quantile(0.95), global_p5=lambda x: x.quantile(0.05))
    .reset_index()
)

# 2) Filter metric.csv to the incident time window in UTC+8
start = tz.localize(pd.to_datetime("2021-03-12 16:00:00"))
end = tz.localize(pd.to_datetime("2021-03-12 16:30:00"))
df_window = df_metric[(df_metric['timestamp_dt'] >= start) & (df_metric['timestamp_dt'] <= end)].copy()

# 3) For each (cmdb_id, kpi_name) present in that window, compute required stats
# Merge global thresholds into windowed data (thresholds computed before filtering)
merged = df_window.merge(global_thresh, on=['cmdb_id', 'kpi_name'], how='left')

# mark anomalies using global thresholds
merged['is_anomaly'] = (merged['value'] >= merged['global_p95']) | (merged['value'] <= merged['global_p5'])

# aggregate window stats
window_stats = (
    merged
    .groupby(['cmdb_id', 'kpi_name'])
    .agg(
        total_points_window=('value', 'count'),
        anomaly_count=('is_anomaly', 'sum'),
        max_value_window=('value', 'max'),
        min_value_window=('value', 'min'),
    )
    .reset_index()
)

# earliest anomaly timestamp per group (if any)
earliest_anom = (
    merged[merged['is_anomaly']]
    .groupby(['cmdb_id', 'kpi_name'])['timestamp_dt']
    .min()
    .reset_index(name='earliest_anomaly_timestamp')
)

# combine
results = window_stats.merge(global_thresh, on=['cmdb_id', 'kpi_name'], how='left').merge(earliest_anom, on=['cmdb_id', 'kpi_name'], how='left')

# format earliest timestamp as string in UTC+8, keep NaT as None
results['earliest_anomaly_timestamp'] = results['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')

# round numeric columns for compactness
for c in ['max_value_window', 'min_value_window', 'global_p95', 'global_p5']:
    results[c] = results[c].round(6)

# 4) Return top 20 sorted by anomaly_count desc; if all zeros then sort by max_value_window desc
if results['anomaly_count'].sum() == 0:
    top20_results = results.sort_values('max_value_window', ascending=False).head(20)
else:
    top20_results = results.sort_values(['anomaly_count', 'total_points_window'], ascending=[False, False]).head(20)

# keep only requested columns in compact form
top20_results = top20_results[
    ['cmdb_id', 'kpi_name', 'total_points_window', 'anomaly_count', 'earliest_anomaly_timestamp',
     'max_value_window', 'min_value_window', 'global_p95', 'global_p5']
].reset_index(drop=True)

top20_results

```
Out[2]:
```
# Summarize the analysis results (use existing `top20_results` from kernel)
num_rows = top20_results.shape[0]
total_anomalies = int(top20_results['anomaly_count'].sum())
rows_all_anom = (top20_results['anomaly_count'] == top20_results['total_points_window']).sum()
distinct_cmdbs = top20_results['cmdb_id'].nunique()
sample_cmdbs = ", ".join(top20_results['cmdb_id'].unique()[:6])
sample_kpis = ", ".join(top20_results['kpi_name'].unique()[:6])
earliest_vals = top20_results['earliest_anomaly_timestamp'].unique()

summary = (
    f"Summary of anomaly analysis for March 12, 2021 16:00–16:30 (UTC+8):\n"
    f"- Returned rows: {num_rows} (top 20)\n"
    f"- Total anomaly points across these rows: {total_anomalies}\n"
    f"- {rows_all_anom} of the {num_rows} rows have anomaly_count == total_points_window (every point flagged as anomaly)\n"
    f"- Distinct cmdb_id values in top results: {distinct_cmdbs} (examples: {sample_cmdbs})\n"
    f"- Example kpi_names in top results: {sample_kpis}\n"
    f"- Earliest anomaly timestamps observed in these rows (UTC+8): {list(earliest_vals)[:3]}\n\n"
    f"Observation: For the top rows (e.g., many JVM-Memory and OSLinux network/process KPIs on IG01, IG02, MG01, MG02, Mysql01),\n"
    f"every sample in the 31-point window was marked anomalous because the window values equal the computed global_p95 or global_p5\n"
    f"(global_p95 and global_p5 in these rows are identical to the observed values), causing the anomaly condition (>= global_p95 or <= global_p5)\n"
    f"to be true for all points. No per-series raw time series are shown here; refer to `top20_results` for the compact table."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  total_points_window  anomaly_count earliest_anomaly_timestamp  max_value_window  min_value_window    global_p95     global_p5
0      IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax                   31             31   2021-03-12 16:00:00+0800      2.040136e+09      2.040136e+09  2.040136e+09  2.040136e+09
1      IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed                   31             31   2021-03-12 16:00:00+0800      5.788098e+07      5.788091e+07  5.788098e+07  5.788091e+07
2      IG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT                   31             31   2021-03-12 16:00:00+0800      0.000000e+00      0.000000e+00  0.000000e+00  0.000000e+00
3      IG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT                   31             31   2021-03-12 16:00:00+0800      0.000000e+00      0.000000e+00  0.000000e+00  0.000000e+00
4      IG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies                   31             31   2021-03-12 16:00:00+0800      1.000000e+00      1.000000e+00  1.000000e+00  1.000000e+00
5      IG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...                   31             31   2021-03-12 16:00:00+0800      3.000000e-01      3.000000e-01  3.000000e-01  3.000000e-01
6      IG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount                   31             31   2021-03-12 16:00:00+0800      3.000000e+00      3.000000e+00  3.000000e+00  3.000000e+00
7      IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax                   31             31   2021-03-12 16:00:00+0800      2.040136e+09      2.040136e+09  2.040136e+09  2.040136e+09
8      IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed                   31             31   2021-03-12 16:00:00+0800      5.788813e+07      5.788813e+07  5.788813e+07  5.788813e+07
9      IG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT                   31             31   2021-03-12 16:00:00+0800      0.000000e+00      0.000000e+00  0.000000e+00  0.000000e+00
10     IG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT                   31             31   2021-03-12 16:00:00+0800      0.000000e+00      0.000000e+00  0.000000e+00  0.000000e+00
11     IG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies                   31             31   2021-03-12 16:00:00+0800      1.000000e+00      1.000000e+00  1.000000e+00  1.000000e+00
12     IG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...                   31             31   2021-03-12 16:00:00+0800      4.000000e-01      3.000000e-01  3.500000e-01  3.000000e-01
13     IG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount                   31             31   2021-03-12 16:00:00+0800      3.000000e+00      3.000000e+00  3.000000e+00  3.000000e+00
14     MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax                   31             31   2021-03-12 16:00:00+0800      2.040136e+09      2.040136e+09  2.040136e+09  2.040136e+09
15     MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed                   31             31   2021-03-12 16:00:00+0800      5.069495e+08      5.069495e+08  5.069495e+08  5.069495e+08
16     MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax                   31             31   2021-03-12 16:00:00+0800      2.040136e+09      2.040136e+09  2.040136e+09  2.040136e+09
17     MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed                   31             31   2021-03-12 16:00:00+0800      9.471521e+08      9.471521e+08  9.471521e+08  9.471521e+08
18  Mysql01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT                   31             31   2021-03-12 16:00:00+0800      0.000000e+00      0.000000e+00  0.000000e+00  0.000000e+00
19  Mysql01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT                   31             31   2021-03-12 16:00:00+0800      0.000000e+00      0.000000e+00  0.000000e+00  0.000000e+00```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

# reuse existing top20_results from kernel
try:
    top20_results
except NameError:
    raise NameError("top20_results not found in kernel. Run the previous analysis cell first.")

tz = pytz.timezone('Asia/Shanghai')

# 1) Filter non-constant series with anomalies
filtered = top20_results[
    (top20_results['max_value_window'] != top20_results['min_value_window']) &
    (top20_results['anomaly_count'] > 0)
].copy()

# sort by anomaly_count desc and limit to top 20
filtered = filtered.sort_values(['anomaly_count', 'total_points_window'], ascending=[False, False]).head(20)

# 2) If filtered set is empty, fallback to top20 rows with anomaly_count>0 sorted by range desc
if filtered.shape[0] == 0:
    fallback = top20_results[top20_results['anomaly_count'] > 0].copy()
    if fallback.shape[0] > 0:
        fallback['range'] = fallback['max_value_window'] - fallback['min_value_window']
        filtered = fallback.sort_values('range', ascending=False).head(20).drop(columns=['range'])
    else:
        # nothing to return
        filtered = top20_results.iloc[0:0].copy()

# Keep only requested columns and ensure compact formatting
cols = ['cmdb_id', 'kpi_name', 'total_points_window', 'anomaly_count',
        'earliest_anomaly_timestamp', 'max_value_window', 'min_value_window', 'global_p95', 'global_p5']
filtered = filtered[cols].reset_index(drop=True)

# 3) distinct cmdb_id values present and their counts
cmdb_counts_filtered = (
    filtered['cmdb_id']
    .value_counts()
    .rename_axis('cmdb_id')
    .reset_index(name='count')
)

# 4) earliest anomaly timestamp per cmdb_id (one row per cmdb_id)
# parse timestamps (handle possible NaN/None)
if filtered.shape[0] > 0:
    ts = pd.to_datetime(filtered['earliest_anomaly_timestamp'], errors='coerce')
    # ensure timezone-aware in Asia/Shanghai
    # if tz-naive, localize; if tz-aware, convert
    def ensure_tz(s):
        if pd.isna(s):
            return s
        if s.tzinfo is None:
            return tz.localize(s)
        else:
            return s.tz_convert(tz)
    ts = ts.apply(ensure_tz)
    filtered['_ts_parsed'] = ts
    earliest_per_cmdb = (
        filtered.dropna(subset=['_ts_parsed'])
        .groupby('cmdb_id')['_ts_parsed']
        .min()
        .reset_index()
    )
    # format back to compact string in UTC+8
    earliest_per_cmdb['_earliest'] = earliest_per_cmdb['_ts_parsed'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
    earliest_per_cmdb = earliest_per_cmdb[['cmdb_id', '_earliest']].rename(columns={'_earliest': 'earliest_anomaly_timestamp'}).reset_index(drop=True)
    # drop helper column
    filtered = filtered.drop(columns=['_ts_parsed'])
else:
    earliest_per_cmdb = filtered[['cmdb_id']].drop_duplicates().assign(earliest_anomaly_timestamp=pd.Series(dtype=str))

# Final compact outputs (limited to top 20 rows already)
filtered, cmdb_counts_filtered, earliest_per_cmdb

```
Out[3]:
```
# Summarize the filtered results in plain English using existing kernel variables
try:
    filtered
    cmdb_counts_filtered
    earliest_per_cmdb
except NameError:
    raise NameError("Required variables (filtered, cmdb_counts_filtered, earliest_per_cmdb) not found in kernel.")

rows = []
for r in filtered.itertuples(index=False):
    # format numeric values compactly
    max_v = f"{r.max_value_window:,}" if float(r.max_value_window).is_integer() else f"{r.max_value_window:.6f}"
    min_v = f"{r.min_value_window:,}" if float(r.min_value_window).is_integer() else f"{r.min_value_window:.6f}"
    gp95 = f"{r.global_p95:,}" if float(r.global_p95).is_integer() else f"{r.global_p95:.6f}"
    gp5  = f"{r.global_p5:,}"  if float(r.global_p5).is_integer()  else f"{r.global_p5:.6f}"
    rows.append(
        f"- {r.cmdb_id} | {r.kpi_name} | points={r.total_points_window}, anomalies={r.anomaly_count}, "
        f"earliest={r.earliest_anomaly_timestamp}, max={max_v}, min={min_v}, global_p95={gp95}, global_p5={gp5}"
    )

cmdb_list = ", ".join(f"{row.cmdb_id}({int(row.count)})" for row in cmdb_counts_filtered.itertuples(index=False))

summary = (
    f"Filtered non-constant series with anomalies: {len(filtered)} row(s).\n"
    f"CMDB counts in this set: {cmdb_list}.\n\n"
    f"Details:\n" + "\n".join(rows) + "\n\n"
    f"Observation: For these rows every point in the 31-point window was flagged anomalous (anomaly_count == total_points_window)."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(  cmdb_id                                           kpi_name  total_points_window  anomaly_count earliest_anomaly_timestamp  max_value_window  min_value_window   global_p95   global_p5
0    IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed                   31             31   2021-03-12 16:00:00+0800        57880976.0        57880912.0  57880976.00  57880912.0
1    IG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...                   31             31   2021-03-12 16:00:00+0800               0.4               0.3         0.35         0.3,   cmdb_id  count
0    IG01      1
1    IG02      1,   cmdb_id earliest_anomaly_timestamp
0    IG01   2021-03-12 16:00:00+0800
1    IG02   2021-03-12 16:00:00+0800)```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# reuse df_trace if present, otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")

# ensure numeric value and drop NaNs
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
df_trace = df_trace.dropna(subset=['value'])

# timezone setup for UTC+8
tz = pytz.timezone('Asia/Shanghai')

# ensure timestamp_dt column exists and is timezone-aware in UTC+8
if 'timestamp_dt' not in df_trace.columns:
    df_trace['timestamp_dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # if exists but naive, localize; if tz-aware but not Asia/Shanghai, convert
    ts = pd.to_datetime(df_trace['timestamp_dt'], errors='coerce')
    if ts.dt.tz is None:
        df_trace['timestamp_dt'] = ts.dt.tz_localize(tz)
    else:
        df_trace['timestamp_dt'] = ts.dt.tz_convert(tz)

# define window in UTC+8
start = tz.localize(pd.to_datetime("2021-03-12 16:00:00"))
end = tz.localize(pd.to_datetime("2021-03-12 16:30:00"))

# filter to window and cmdb_id IG01 and IG02
df_window_trace = df_trace[
    (df_trace['timestamp_dt'] >= start) &
    (df_trace['timestamp_dt'] <= end) &
    (df_trace['cmdb_id'].isin(['IG01', 'IG02']))
].copy()

# if no data, produce empty results gracefully
if df_window_trace.shape[0] == 0:
    top20_table = df_window_trace.head(0)
    cmdb_list_df = pd.DataFrame(columns=['cmdb_id', 'count'])
    trace_counts_top50 = pd.DataFrame(columns=['trace_name', 'count'])
else:
    # aggregate per (cmdb_id, trace_name)
    agg = (
        df_window_trace
        .groupby(['cmdb_id', 'trace_name'])
        .agg(
            count_in_window=('value', 'count'),
            earliest_timestamp=('timestamp_dt', 'min'),
            latest_timestamp=('timestamp_dt', 'max'),
            mean_value=('value', 'mean'),
            max_value=('value', 'max'),
            p95_value=('value', lambda x: x.quantile(0.95))
        )
        .reset_index()
    )

    # format timestamps to compact strings in UTC+8
    agg['earliest_timestamp'] = agg['earliest_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
    agg['latest_timestamp'] = agg['latest_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')

    # round numeric columns for compact display
    for c in ['mean_value', 'max_value', 'p95_value']:
        agg[c] = agg[c].round(6)

    # sort by count desc and limit to top 20
    top20_table = agg.sort_values('count_in_window', ascending=False).head(20).reset_index(drop=True)

    # distinct cmdb_id values found and their counts
    cmdb_list_df = (
        df_window_trace['cmdb_id']
        .value_counts()
        .rename_axis('cmdb_id')
        .reset_index(name='count')
    )

    # distinct trace_name values with counts, limited to top 50 by count
    trace_counts_top50 = (
        df_window_trace['trace_name']
        .value_counts()
        .rename_axis('trace_name')
        .reset_index(name='count')
        .head(50)
    )

# Final compact outputs
top20_table, cmdb_list_df, trace_counts_top50

```
Out[4]:
```
```python
# Summarize trace window analysis using existing kernel variables: top20_table, cmdb_list_df, trace_counts_top50
try:
    top20_table
    cmdb_list_df
    trace_counts_top50
except NameError:
    raise NameError("Required variables (top20_table, cmdb_list_df, trace_counts_top50) not found in kernel.")

n_rows = int(top20_table.shape[0])
cmdbs = ", ".join(f"{r.cmdb_id}({int(r.count)})" for r in cmdb_list_df.itertuples(index=False))
n_trace_names_reported = int(trace_counts_top50.shape[0])
top_trace_examples = ", ".join(trace_counts_top50['trace_name'].head(5).tolist())

# compute overall earliest and latest timestamps in top20_table (they are strings like 'YYYY-MM-DD HH:MM:SS+0800')
earliest = top20_table['earliest_timestamp'].min() if n_rows>0 else None
latest = top20_table['latest_timestamp'].max() if n_rows>0 else None

# pick a few notable rows for short examples
examples = []
for r in top20_table.head(3).itertuples(index=False):
    examples.append(f"{r.cmdb_id}:{r.trace_name} (count={int(r.count_in_window)}, mean={r.mean_value}, p95={r.p95_value}, max={r.max_value})")

summary = (
    f"Trace window summary (2021-03-12 16:00–16:30 UTC+8) limited to IG01 and IG02:\n"
    f"- Returned groups: {n_rows} (top 20 by count).\n"
    f"- CMDBs found and point counts: {cmdbs}.\n"
    f"- Trace names reported (top {n_trace_names_reported} by count): {top_trace_examples}.\n"
    f"- Window span across returned groups: earliest={earliest}, latest={latest} (UTC+8).\n"
    f"- Most groups have 30 points in-window (per-group counts are typically 30). Examples:\n  - " + "\n  - ".join(examples) + "\n"
    f"- Observations: For duration metrics (e.g., trace.self.duration_mean and *_duration_p95) p95 values are substantially higher than means\n"
    f"  (example IG01 trace.self.duration_p95 mean={top20_table.loc[1,'mean_value']} p95={top20_table.loc[1,'p95_value']}).\n"
    f"- No raw time series shown; see `top20_table`, `cmdb_list_df`, and `trace_counts_top50` for compact tables."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                       trace_name  count_in_window        earliest_timestamp          latest_timestamp   mean_value    max_value    p95_value
0     IG01         trace.self.duration_mean               30  2021-03-12 16:00:00+0800  2021-03-12 16:29:00+0800     0.681692     3.139906     2.097008
1     IG01          trace.self.duration_p95               30  2021-03-12 16:00:00+0800  2021-03-12 16:29:00+0800     2.981140    10.179150     9.594832
2     IG01             trace.self.row_count               30  2021-03-12 16:00:00+0800  2021-03-12 16:29:00+0800  1718.866667  1988.000000  1945.900000
3     IG01  trace.to_Tomcat01.duration_mean               30  2021-03-12 16:00:00+0800  2021-03-12 16:29:00+0800     0.687338     2.979292     2.164947
4     IG01   trace.to_Tomcat01.duration_p95               30  2021-03-12 16:00:00+0800  2021-03-12 16:29:00+0800     2.981180    11.930500     9.184085
5     IG01      trace.to_Tomcat01.row_count               30  2021-03-12 16:00:00+0800  2021-03-12 16:29:00+0800   216.133333   276.000000   256.350000
6     IG01  trace.to_Tomcat02.duration_mean               30  2021-03-12 16:00:00+0800  2021-03-12 16:29:00+0800     0.707790     3.247291     2.449984
7     IG01   trace.to_Tomcat02.duration_p95               30  2021-03-12 16:00:00+0800  2021-03-12 16:29:00+0800     3.102078    12.245000     9.873520
8     IG01      trace.to_Tomcat02.row_count               30  2021-03-12 16:00:00+0800  2021-03-12 16:29:00+0800   208.900000   268.000000   244.950000
9     IG01  trace.to_Tomcat03.duration_mean               30  2021-03-12 16:00:00+0800  2021-03-12 16:29:00+0800     0.657340     2.915092     1.836740
10    IG01   trace.to_Tomcat03.duration_p95               30  2021-03-12 16:00:00+0800  2021-03-12 16:29:00+0800     3.034898    11.448000     9.617235
11    IG01      trace.to_Tomcat03.row_count               30  2021-03-12 16:00:00+0800  2021-03-12 16:29:00+0800   214.600000   254.000000   246.750000
12    IG01  trace.to_Tomcat04.duration_mean               30  2021-03-12 16:00:00+0800  2021-03-12 16:29:00+0800     0.667289     3.423783     1.933611
13    IG01   trace.to_Tomcat04.duration_p95               30  2021-03-12 16:00:00+0800  2021-03-12 16:29:00+0800     2.826502    11.021000     8.587547
14    IG01      trace.to_Tomcat04.row_count               30  2021-03-12 16:00:00+0800  2021-03-12 16:29:00+0800   219.800000   259.000000   256.000000
15    IG02         trace.self.duration_mean               30  2021-03-12 16:00:00+0800  2021-03-12 16:29:00+0800     0.671753     3.042090     2.028319
16    IG02          trace.self.duration_p95               30  2021-03-12 16:00:00+0800  2021-03-12 16:29:00+0800     2.933000    10.199000     9.276135
17    IG02             trace.self.row_count               30  2021-03-12 16:00:00+0800  2021-03-12 16:29:00+0800  1716.600000  2188.000000  1888.100000
18    IG02  trace.to_Tomcat01.duration_mean               30  2021-03-12 16:00:00+0800  2021-03-12 16:29:00+0800     0.667252     3.104602     1.968169
19    IG02   trace.to_Tomcat01.duration_p95               30  2021-03-12 16:00:00+0800  2021-03-12 16:29:00+0800     2.807545    10.065650     9.213750,   cmdb_id  count
0    IG01    450
1    IG02    450,                          trace_name  count
0          trace.self.duration_mean     60
1           trace.self.duration_p95     60
2              trace.self.row_count     60
3   trace.to_Tomcat01.duration_mean     60
4    trace.to_Tomcat01.duration_p95     60
5       trace.to_Tomcat01.row_count     60
6   trace.to_Tomcat02.duration_mean     60
7    trace.to_Tomcat02.duration_p95     60
8       trace.to_Tomcat02.row_count     60
9   trace.to_Tomcat03.duration_mean     60
10   trace.to_Tomcat03.duration_p95     60
11      trace.to_Tomcat03.row_count     60
12  trace.to_Tomcat04.duration_mean     60
13   trace.to_Tomcat04.duration_p95     60
14      trace.to_Tomcat04.row_count     60)```
```

In [ ]:
# In[5]:

import pandas as pd
import re
import pytz

# reuse df_metric if present, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

# ensure value is numeric and drop NaNs (reuse existing cleaned df if applicable)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_metric = df_metric.dropna(subset=['value'])

# focus on cmdb_id IG01 and IG02 across the full available series (no time filtering for thresholds/counts)
cmdb_filter = ['IG01', 'IG02']
df_ig = df_metric[df_metric['cmdb_id'].isin(cmdb_filter)].copy()

# 1) distinct kpi_name values and their sample counts per cmdb_id (full series)
kpi_counts = (
    df_ig
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .count()
    .reset_index(name='count')
)

# top 50 kpi_name entries sorted by count descending (compact)
top50_kpi_counts = kpi_counts.sort_values('count', ascending=False).head(50).reset_index(drop=True)

# 2) resource-relevant KPI substring search (case-insensitive)
substrs = ['CPU', 'MEMORY', 'Memory', 'JVM', 'Heap', 'Network', 'NET', 'DISK', 'IO', 'OOM', 'gc', 'thread']
# build case-insensitive regex (escape substrings where needed)
pattern = "(" + "|".join(re.escape(s) for s in substrs) + ")"
mask_resource = df_ig['kpi_name'].astype(str).str.contains(pattern, case=False, na=False, regex=True)

resource_kpis = df_ig[mask_resource].copy()

# aggregated short list per cmdb_id: kpi_name and counts (limited to top 50 rows overall for compactness)
resource_kpi_counts = (
    resource_kpis
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .count()
    .reset_index(name='count')
    .sort_values(['cmdb_id', 'count'], ascending=[True, False])
    .reset_index(drop=True)
)

resource_kpi_counts_top50 = resource_kpi_counts.head(50)

# also provide a compact per-cmdb summary of how many matched resource-relevant KPIs
resource_summary_per_cmdb = (
    resource_kpi_counts
    .groupby('cmdb_id')['kpi_name']
    .nunique()
    .reset_index(name='distinct_resource_kpi_count')
)

# Final compact outputs: top50_kpi_counts, resource_kpi_counts_top50, resource_summary_per_cmdb
top50_kpi_counts, resource_kpi_counts_top50, resource_summary_per_cmdb

```
Out[5]:
```
```python
# Summarize findings in plain English using existing kernel variables:
# top50_kpi_counts, resource_kpi_counts_top50, resource_summary_per_cmdb

try:
    top50_kpi_counts
    resource_kpi_counts_top50
    resource_summary_per_cmdb
except NameError:
    raise NameError("Expected variables (top50_kpi_counts, resource_kpi_counts_top50, resource_summary_per_cmdb) not found in kernel.")

# prepare short examples of top KPIs per cmdb in the top-50 list
examples_per_cmdb = {}
for cmdb, grp in top50_kpi_counts.groupby('cmdb_id'):
    top_names = grp.head(5)['kpi_name'].tolist()
    examples_per_cmdb[cmdb] = top_names

# prepare a few resource-relevant KPI examples per cmdb (from resource_kpi_counts_top50)
resource_examples = {}
for cmdb, grp in resource_kpi_counts_top50.groupby('cmdb_id'):
    resource_examples[cmdb] = grp.head(5)['kpi_name'].tolist()

summary = (
    "Summary of metric KPI coverage for IG01 and IG02 (full series):\n"
    f"- Top KPI samples (top 50 by count) include many JVM memory/thread and OS CPU/memory metrics. Examples per host:\n"
    f"  - IG01 examples: {examples_per_cmdb.get('IG01', [])}\n"
    f"  - IG02 examples: {examples_per_cmdb.get('IG02', [])}\n"
    f"- Sample counts: the highest-count KPIs (e.g., JVM-Memory_... and JVM-Threads_... and OSLinux-CPU_CPU_CPULoad) have 31 samples each.\n"
    f"- Resource-relevant KPI match counts (case-insensitive substrings like CPU/MEMORY/JVM/Heap/Network/DISK/IO/OOM/gc/thread):\n"
    f"  - IG01: {int(resource_summary_per_cmdb.loc[resource_summary_per_cmdb['cmdb_id']=='IG01','distinct_resource_kpi_count'].values[0]) if ('IG01' in resource_summary_per_cmdb['cmdb_id'].values) else 0} distinct matches (examples: {resource_examples.get('IG01', [])})\n"
    f"  - IG02: {int(resource_summary_per_cmdb.loc[resource_summary_per_cmdb['cmdb_id']=='IG02','distinct_resource_kpi_count'].values[0]) if ('IG02' in resource_summary_per_cmdb['cmdb_id'].values) else 0} distinct matches (examples: {resource_examples.get('IG02', [])})\n"
    f"- Recommendation: Use the listed JVM, CPU, MEMORY, Heap, and Network KPIs (examples above) as resource-focused candidates for deeper anomaly or threshold analysis.\n"
    "Note: Counts are across the full available series (no time filtering). See `top50_kpi_counts` and `resource_kpi_counts_top50` for compact tables."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                                      kpi_name  count
0     IG01      JVM-Memory_7778_JVM_Memory_HeapMemoryMax     31
1     IG01     JVM-Memory_7778_JVM_Memory_HeapMemoryUsed     31
2     IG01   JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31
3     IG01      JVM-Threads_7778_JVM_ThreadCount_Threads     31
4     IG01                       OSLinux-CPU_CPU_CPULoad     31
..     ...                                           ...    ...
45    IG02      OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem     20
46    IG01  OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc     20
47    IG01  OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc     20
48    IG02         OSLinux-OSLinux_MEMORY_MEMORY_UserMem     20
49    IG02        OSLinux-OSLinux_MEMORY_MEMORY_CacheMem     20

[50 rows x 3 columns],    cmdb_id                                         kpi_name  count
0     IG01         JVM-Memory_7778_JVM_Memory_HeapMemoryMax     31
1     IG01        JVM-Memory_7778_JVM_Memory_HeapMemoryUsed     31
2     IG01      JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31
3     IG01         JVM-Threads_7778_JVM_ThreadCount_Threads     31
4     IG01                          OSLinux-CPU_CPU_CPULoad     31
..     ...                                              ...    ...
45    IG01  OSLinux-OSLinux_NETWORK_ens160_NETBandwidthUtil     10
46    IG01          OSLinux-OSLinux_NETWORK_ens160_NETInErr     10
47    IG01         OSLinux-OSLinux_NETWORK_ens160_NETOutErr     10
48    IG01                  OSLinux-CPU_CPU-0_SingleCpuUtil      9
49    IG01                  OSLinux-CPU_CPU-2_SingleCpuUtil      9

[50 rows x 3 columns],   cmdb_id  distinct_resource_kpi_count
0    IG01                           52
1    IG02                           52)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz
import re

# reuse df_metric if present, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

# ensure numeric 'value' and drop NaNs
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_metric = df_metric.dropna(subset=['value'])

# ensure timezone-aware timestamp in UTC+8 (Asia/Shanghai)
tz = pytz.timezone('Asia/Shanghai')
if 'timestamp_dt' not in df_metric.columns:
    df_metric['timestamp_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # normalize existing column to tz-aware Asia/Shanghai
    tmp = pd.to_datetime(df_metric['timestamp_dt'], errors='coerce')
    if tmp.dt.tz is None:
        df_metric['timestamp_dt'] = tmp.dt.tz_localize(tz)
    else:
        df_metric['timestamp_dt'] = tmp.dt.tz_convert(tz)

# parameters
cmdbs = ['IG01', 'IG02']
start = tz.localize(pd.to_datetime("2021-03-12 16:00:00"))
end = tz.localize(pd.to_datetime("2021-03-12 16:30:00"))

# 1a) Compute global p95 and p5 per (cmdb_id, kpi_name) for IG01 and IG02 using full series (no time filter)
df_ig_full = df_metric[df_metric['cmdb_id'].isin(cmdbs)].copy()
global_thresh = (
    df_ig_full
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(global_p95=lambda x: x.quantile(0.95), global_p5=lambda x: x.quantile(0.05))
    .reset_index()
)

# 1b) Filter metric.csv to the incident window (UTC+8)
df_window = df_ig_full[(df_ig_full['timestamp_dt'] >= start) & (df_ig_full['timestamp_dt'] <= end)].copy()

# 1c) Merge thresholds into windowed data then compute per-group stats
merged = df_window.merge(global_thresh, on=['cmdb_id', 'kpi_name'], how='left')
merged['is_anomaly'] = (merged['value'] >= merged['global_p95']) | (merged['value'] <= merged['global_p5'])

# aggregate stats in window
window_stats = (
    merged
    .groupby(['cmdb_id', 'kpi_name'])
    .agg(
        total_points_window=('value', 'count'),
        anomaly_count=('is_anomaly', 'sum'),
        max_value_window=('value', 'max'),
        min_value_window=('value', 'min'),
    )
    .reset_index()
)

# earliest anomaly timestamp per group
earliest_anom = (
    merged[merged['is_anomaly']]
    .groupby(['cmdb_id', 'kpi_name'])['timestamp_dt']
    .min()
    .reset_index(name='earliest_anomaly_timestamp')
)

# combine results
results = window_stats.merge(global_thresh, on=['cmdb_id', 'kpi_name'], how='left').merge(earliest_anom, on=['cmdb_id', 'kpi_name'], how='left')

# format earliest timestamp as string (UTC+8) and round numeric columns for compactness
results['earliest_anomaly_timestamp'] = results['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
for c in ['max_value_window', 'min_value_window', 'global_p95', 'global_p5']:
    results[c] = results[c].round(6)

# 1d) Filter to anomaly_count>0 AND non-constant series (max != min)
filtered_results = results[(results['anomaly_count'] > 0) & (results['max_value_window'] != results['min_value_window'])].copy()

# 1e) Top 20 rows sorted by anomaly_count desc (tie-breaker total_points_window desc)
metric_anom_filtered_top20 = filtered_results.sort_values(['anomaly_count', 'total_points_window'], ascending=[False, False]).head(20).reset_index(drop=True)

# Keep only requested columns in compact form
metric_anom_filtered_top20 = metric_anom_filtered_top20[
    ['cmdb_id', 'kpi_name', 'total_points_window', 'anomaly_count', 'earliest_anomaly_timestamp',
     'max_value_window', 'min_value_window', 'global_p95', 'global_p5']
].copy()

# 1f) Prepare three small lists (top 10 each) from the filtered set by case-insensitive patterns
def top_n_by_pattern(df, pattern, n=10):
    mask = df['kpi_name'].str.contains(pattern, case=False, na=False, regex=True)
    return df[mask].sort_values(['anomaly_count', 'total_points_window'], ascending=[False, False]).head(n).reset_index(drop=True)

jvm_pattern = r'JVM|Heap|gc'
cpu_pattern = r'CPU|SingleCpu|cpu'
net_pattern = r'NET|Network|ens|InErr|OutErr'

jvm_related_top10 = top_n_by_pattern(metric_anom_filtered_top20, jvm_pattern, 10)
cpu_related_top10 = top_n_by_pattern(metric_anom_filtered_top20, cpu_pattern, 10)
net_related_top10 = top_n_by_pattern(metric_anom_filtered_top20, net_pattern, 10)

# 2) Process log.csv and error_logs.csv for same window and cmdbs
# Load and prepare log.csv
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")

# ensure timestamp and timezone
df_log['timestamp'] = pd.to_numeric(df_log['timestamp'], errors='coerce')
df_log = df_log.dropna(subset=['timestamp'])
df_log['timestamp_dt'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

df_log_window = df_log[(df_log['timestamp_dt'] >= start) & (df_log['timestamp_dt'] <= end) & (df_log['cmdb_id'].isin(cmdbs))].copy()

# For log.csv: total record count in-window per cmdb_id
log_counts_df = df_log_window.groupby('cmdb_id').size().rename('count').reset_index()

# up to 10 sample rows per cmdb_id: select columns timestamp, cmdb_id, log_name
log_samples_list = []
for cm in cmdbs:
    sub = df_log_window[df_log_window['cmdb_id'] == cm].copy()
    if sub.shape[0] == 0:
        # explicit note DataFrame row
        note = pd.DataFrame([{'timestamp': None, 'cmdb_id': cm, 'log_name': f'No records in-window for {cm} (log.csv)'}])
        log_samples_list.append(note)
    else:
        samp = sub.sort_values('timestamp_dt').head(10)[['timestamp_dt', 'cmdb_id', 'log_name']].copy()
        samp['timestamp_dt'] = samp['timestamp_dt'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
        samp = samp.rename(columns={'timestamp_dt': 'timestamp'})
        log_samples_list.append(samp)
log_samples_df = pd.concat(log_samples_list, ignore_index=True)

# Load and prepare error_logs.csv
try:
    df_error
except NameError:
    df_error = pd.read_csv("error_logs.csv")

# ensure timestamp and timezone
df_error['timestamp'] = pd.to_numeric(df_error['timestamp'], errors='coerce')
df_error = df_error.dropna(subset=['timestamp'])
df_error['timestamp_dt'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

df_error_window = df_error[(df_error['timestamp_dt'] >= start) & (df_error['timestamp_dt'] <= end) & (df_error['cmdb_id'].isin(cmdbs))].copy()

# For error_logs.csv: total record count in-window per cmdb_id
error_counts_df = df_error_window.groupby('cmdb_id').size().rename('count').reset_index()

# up to 10 sample rows per cmdb_id: select columns timestamp, cmdb_id, message
error_samples_list = []
for cm in cmdbs:
    sub = df_error_window[df_error_window['cmdb_id'] == cm].copy()
    if sub.shape[0] == 0:
        note = pd.DataFrame([{'timestamp': None, 'cmdb_id': cm, 'message': f'No records in-window for {cm} (error_logs.csv)'}])
        error_samples_list.append(note)
    else:
        samp = sub.sort_values('timestamp_dt').head(10)[['timestamp_dt', 'cmdb_id', 'message']].copy()
        samp['timestamp_dt'] = samp['timestamp_dt'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
        samp = samp.rename(columns={'timestamp_dt': 'timestamp'})
        error_samples_list.append(samp)
error_samples_df = pd.concat(error_samples_list, ignore_index=True)

# If a cmdb_id is missing in counts, add explicit note rows to counts DataFrames for clarity
for cm in cmdbs:
    if cm not in log_counts_df['cmdb_id'].values:
        log_counts_df = pd.concat([log_counts_df, pd.DataFrame([{'cmdb_id': cm, 'count': 0}])], ignore_index=True)
    if cm not in error_counts_df['cmdb_id'].values:
        error_counts_df = pd.concat([error_counts_df, pd.DataFrame([{'cmdb_id': cm, 'count': 0}])], ignore_index=True)

# keep counts sorted by cmdb order
log_counts_df = log_counts_df.set_index('cmdb_id').reindex(cmdbs).reset_index()
error_counts_df = error_counts_df.set_index('cmdb_id').reindex(cmdbs).reset_index()

# Final compact outputs (store intermediate full tables in variables as well)
metric_anom_filtered_top20, jvm_related_top10, cpu_related_top10, net_related_top10, log_counts_df, log_samples_df, error_counts_df, error_samples_df

```
Out[6]:
```
# Summarize findings in plain English using existing kernel variables (displayed as a single compact string)
try:
    metric_anom_filtered_top20
    jvm_related_top10
    cpu_related_top10
    net_related_top10
    log_counts_df
    log_samples_df
    error_counts_df
    error_samples_df
except NameError:
    raise NameError("Expected analysis variables not found in kernel.")

num_rows = int(metric_anom_filtered_top20.shape[0])
total_anomalies = int(metric_anom_filtered_top20['anomaly_count'].sum()) if num_rows>0 else 0

# top 3 example rows
examples = []
for r in metric_anom_filtered_top20.head(3).itertuples(index=False):
    examples.append(f"{r.cmdb_id} | {r.kpi_name} | points={r.total_points_window}, anomalies={r.anomaly_count}, max={r.max_value_window}, min={r.min_value_window}")

jvm_count = int(jvm_related_top10.shape[0])
cpu_count = int(cpu_related_top10.shape[0])
net_count = int(net_related_top10.shape[0])

# log/error counts per cmdb
log_counts = ", ".join(f"{row.cmdb_id}={int(row['count'])}" for _, row in log_counts_df.iterrows())
error_counts = ", ".join(f"{row.cmdb_id}={int(row['count'])}" for _, row in error_counts_df.iterrows())

summary = (
    "Summary of metric and log analysis for IG01/IG02 during 2021-03-12 16:00–16:30 (UTC+8):\n"
    f"- Metric anomalies (non-constant series with >0 anomalies): {num_rows} rows returned, total anomaly points={total_anomalies}.\n"
    f"- Top examples (first 3):\n  - " + "\n  - ".join(examples) + "\n"
    f"- Resource-focused subsets from the filtered set (top ≤10 each): JVM-related={jvm_count} rows, CPU-related={cpu_count} rows, Network-related={net_count} rows.\n"
    f"  - JVM example: {jvm_related_top10['kpi_name'].iat[0] if jvm_count>0 else 'none'}\n"
    f"  - CPU example: {cpu_related_top10['kpi_name'].iat[0] if cpu_count>0 else 'none'}\n"
    f"  - Network example: {net_related_top10['kpi_name'].iat[0] if net_count>0 else 'none'}\n"
    f"- log.csv records in-window per cmdb: {log_counts}.\n"
    f"  - Sample log rows (up to 10 per cmdb) are stored in variable `log_samples_df`.\n"
    f"- error_logs.csv records in-window per cmdb: {error_counts}.\n"
    f"  - Sample error rows (up to 10 per cmdb) are stored in variable `error_samples_df`.\n"
    "Notes:\n"
    "- Global thresholds (p95/p5) were computed using the full series before window filtering.\n"
    "- The metric table excludes constant-series false positives by requiring max != min.\n"
    "- No raw time series are shown here; use the stored variables for compact tables (e.g., `metric_anom_filtered_top20`, `jvm_related_top10`)."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                                           kpi_name  total_points_window  anomaly_count earliest_anomaly_timestamp  max_value_window  min_value_window    global_p95     global_p5
0     IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed                   31             31   2021-03-12 16:00:00+0800      5.788098e+07      5.788091e+07  5.788098e+07  5.788091e+07
1     IG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...                   31             31   2021-03-12 16:00:00+0800      4.000000e-01      3.000000e-01  3.500000e-01  3.000000e-01
2     IG02       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc                   21             18   2021-03-12 16:00:00+0800      9.800000e+01      8.400000e+01  9.800000e+01  8.500000e+01
3     IG01                            OSLinux-CPU_CPU_CPULoad                   31             17   2021-03-12 16:02:00+0800      1.200000e+00      1.000000e+00  1.155000e+00  1.000000e+00
4     IG01    OSLinux-OSLinux_FILESYSTEM_-_FSInodeUsedPercent                   15             15   2021-03-12 16:01:00+0800      1.636640e+01      1.636637e+01  1.636640e+01  1.636637e+01
5     IG01        OSLinux-OSLinux_FILESYSTEM_-boot_FSCapacity                   15             15   2021-03-12 16:01:00+0800      1.541700e+01      1.541695e+01  1.541700e+01  1.541695e+01
6     IG01  OSLinux-OSLinux_FILESYSTEM_-boot_FSInodeUsedPe...                   15             15   2021-03-12 16:01:00+0800      4.715000e-01      4.714970e-01  4.715000e-01  4.714970e-01
7     IG01  OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSCapacity                   15             15   2021-03-12 16:01:00+0800      4.268120e-01      4.268000e-01  4.268120e-01  4.268000e-01
8     IG01  OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSInode...                   15             15   2021-03-12 16:01:00+0800      4.200000e-03      4.196000e-03  4.200000e-03  4.196000e-03
9     IG01  OSLinux-OSLinux_FILESYSTEM_-home_FSInodeUsedPe...                   15             15   2021-03-12 16:01:00+0800      2.121000e-01      2.120970e-01  2.121000e-01  2.120970e-01
10    IG01  OSLinux-OSLinux_FILESYSTEM_-tmp_FSInodeUsedPer...                   15             15   2021-03-12 16:01:00+0800      1.590000e-02      1.586900e-02  1.590000e-02  1.586900e-02
11    IG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps                   15             15   2021-03-12 16:01:00+0800      1.000000e+00      0.000000e+00  1.000000e+00  0.000000e+00
12    IG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead                   15             15   2021-03-12 16:01:00+0800      6.000000e+00      0.000000e+00  6.000000e+00  0.000000e+00
13    IG02    OSLinux-OSLinux_FILESYSTEM_-_FSInodeUsedPercent                   15             15   2021-03-12 16:00:00+0800      1.636620e+01      1.636617e+01  1.636620e+01  1.636617e+01
14    IG02        OSLinux-OSLinux_FILESYSTEM_-boot_FSCapacity                   15             15   2021-03-12 16:00:00+0800      1.541700e+01      1.541695e+01  1.541700e+01  1.541695e+01
15    IG02  OSLinux-OSLinux_FILESYSTEM_-boot_FSInodeUsedPe...                   15             15   2021-03-12 16:00:00+0800      4.715000e-01      4.714970e-01  4.715000e-01  4.714970e-01
16    IG02  OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSCapacity                   15             15   2021-03-12 16:00:00+0800      4.268120e-01      4.268000e-01  4.268120e-01  4.268000e-01
17    IG02  OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSInode...                   15             15   2021-03-12 16:00:00+0800      4.200000e-03      4.196000e-03  4.200000e-03  4.196000e-03
18    IG02  OSLinux-OSLinux_FILESYSTEM_-home_FSInodeUsedPe...                   15             15   2021-03-12 16:00:00+0800      2.290340e-01      2.290000e-01  2.290340e-01  2.290000e-01
19    IG02  OSLinux-OSLinux_FILESYSTEM_-tmp_FSInodeUsedPer...                   15             15   2021-03-12 16:00:00+0800      1.590000e-02      1.586900e-02  1.590000e-02  1.586900e-02,   cmdb_id                                     kpi_name  total_points_window  anomaly_count earliest_anomaly_timestamp  max_value_window  min_value_window  global_p95   global_p5
0    IG01  JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed                   31             31   2021-03-12 16:00:00+0800        57880976.0        57880912.0  57880976.0  57880912.0,   cmdb_id                                           kpi_name  total_points_window  anomaly_count earliest_anomaly_timestamp  max_value_window  min_value_window  global_p95  global_p5
0    IG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...                   31             31   2021-03-12 16:00:00+0800               0.4               0.3       0.350        0.3
1    IG01                            OSLinux-CPU_CPU_CPULoad                   31             17   2021-03-12 16:02:00+0800               1.2               1.0       1.155        1.0, Empty DataFrame
Columns: [cmdb_id, kpi_name, total_points_window, anomaly_count, earliest_anomaly_timestamp, max_value_window, min_value_window, global_p95, global_p5]
Index: [],   cmdb_id  count
0    IG01      0
1    IG02      0,   timestamp cmdb_id                                 log_name
0      None    IG01  No records in-window for IG01 (log.csv)
1      None    IG02  No records in-window for IG02 (log.csv),   cmdb_id  count
0    IG01      0
1    IG02      0,   timestamp cmdb_id                                         message
0      None    IG01  No records in-window for IG01 (error_logs.csv)
1      None    IG02  No records in-window for IG02 (error_logs.csv))```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz
import re

# Reuse existing df_metric if available; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

# Ensure numeric values and drop NaNs
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_metric = df_metric.dropna(subset=['value'])

# Ensure timestamp_dt exists and is timezone-aware in UTC+8 (Asia/Shanghai)
tz = pytz.timezone('Asia/Shanghai')
if 'timestamp_dt' not in df_metric.columns:
    df_metric['timestamp_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    tmp_ts = pd.to_datetime(df_metric['timestamp_dt'], errors='coerce')
    # If tz-naive, localize to Asia/Shanghai; if tz-aware, convert
    if tmp_ts.dt.tz is None:
        # tmp_ts may be timezone-naive but actually in UTC seconds; safer to convert from original 'timestamp'
        df_metric['timestamp_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        df_metric['timestamp_dt'] = tmp_ts.dt.tz_convert(tz)

# Parameters
tomcats = ['Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04']
start = tz.localize(pd.to_datetime("2021-03-12 16:00:00"))
end = tz.localize(pd.to_datetime("2021-03-12 16:30:00"))

# 1) Compute global p95 and p5 per (cmdb_id, kpi_name) using full series for Tomcats
df_tomcat_full = df_metric[df_metric['cmdb_id'].isin(tomcats)].copy()
tomcat_global_thresh = (
    df_tomcat_full
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(global_p95=lambda x: x.quantile(0.95), global_p5=lambda x: x.quantile(0.05))
    .reset_index()
)

# 2) Filter to incident window for Tomcats
df_tomcat_window = df_tomcat_full[(df_tomcat_full['timestamp_dt'] >= start) & (df_tomcat_full['timestamp_dt'] <= end)].copy()

# 3) Merge thresholds then compute window stats and anomalies
merged_tomcat = df_tomcat_window.merge(tomcat_global_thresh, on=['cmdb_id', 'kpi_name'], how='left')
merged_tomcat['is_anomaly'] = (merged_tomcat['value'] >= merged_tomcat['global_p95']) | (merged_tomcat['value'] <= merged_tomcat['global_p5'])

tomcat_window_stats = (
    merged_tomcat
    .groupby(['cmdb_id', 'kpi_name'])
    .agg(
        total_points_window=('value', 'count'),
        anomaly_count=('is_anomaly', 'sum'),
        max_value_window=('value', 'max'),
        min_value_window=('value', 'min')
    )
    .reset_index()
)

earliest_anom_tomcat = (
    merged_tomcat[merged_tomcat['is_anomaly']]
    .groupby(['cmdb_id', 'kpi_name'])['timestamp_dt']
    .min()
    .reset_index(name='earliest_anomaly_timestamp')
)

tomcat_results = (
    tomcat_window_stats
    .merge(tomcat_global_thresh, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(earliest_anom_tomcat, on=['cmdb_id', 'kpi_name'], how='left')
)

# Format and round for compact display
tomcat_results['earliest_anomaly_timestamp'] = tomcat_results['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
for c in ['max_value_window', 'min_value_window', 'global_p95', 'global_p5']:
    tomcat_results[c] = tomcat_results[c].round(6)

# 4) Filter to anomaly_count>0 and non-constant series (max != min)
tomcat_filtered = tomcat_results[(tomcat_results['anomaly_count'] > 0) & (tomcat_results['max_value_window'] != tomcat_results['min_value_window'])].copy()

# 5) Top 20 by anomaly_count desc (tie-breaker: total_points_window desc)
metric_tomcat_anom_top20 = tomcat_filtered.sort_values(['anomaly_count', 'total_points_window'], ascending=[False, False]).head(20).reset_index(drop=True)
metric_tomcat_anom_top20 = metric_tomcat_anom_top20[
    ['cmdb_id', 'kpi_name', 'total_points_window', 'anomaly_count', 'earliest_anomaly_timestamp',
     'max_value_window', 'min_value_window', 'global_p95', 'global_p5']
].copy()

# 6) Prepare three small lists (top 10 each) from filtered set by patterns
def top_n_by_pattern(df, pattern, n=10):
    mask = df['kpi_name'].str.contains(pattern, case=False, na=False, regex=True)
    return df[mask].sort_values(['anomaly_count', 'total_points_window'], ascending=[False, False]).head(n).reset_index(drop=True)

jvm_pattern = r'JVM|Heap|gc'
cpu_pattern = r'CPU|SingleCpu|cpu'
net_pattern = r'NET|Network|ens|InErr|OutErr'

jvm_related_top10 = top_n_by_pattern(tomcat_filtered, jvm_pattern, 10)
cpu_related_top10 = top_n_by_pattern(tomcat_filtered, cpu_pattern, 10)
net_related_top10 = top_n_by_pattern(tomcat_filtered, net_pattern, 10)

# Final compact outputs (stored in kernel variables)
metric_tomcat_anom_top20, jvm_related_top10, cpu_related_top10, net_related_top10

```
Out[7]:
```
# Summarize Tomcat metrics analysis in plain English using kernel variables
try:
    metric_tomcat_anom_top20
    jvm_related_top10
    cpu_related_top10
    net_related_top10
except NameError:
    raise NameError("Expected analysis variables (metric_tomcat_anom_top20, jvm_related_top10, cpu_related_top10, net_related_top10) not found in kernel.")

n_rows = int(metric_tomcat_anom_top20.shape[0])
examples = []
for r in metric_tomcat_anom_top20.head(5).itertuples(index=False):
    examples.append(f"{r.cmdb_id} | {r.kpi_name} | points={r.total_points_window}, anomalies={r.anomaly_count}, max={r.max_value_window}, min={r.min_value_window}, earliest={r.earliest_anomaly_timestamp}")

summary = (
    f"Tomcat services inspection (2021-03-12 16:00–16:30 UTC+8):\n"
    f"- Total anomalous non-constant KPI rows found: {n_rows} (top 20 shown in variable `metric_tomcat_anom_top20`).\n\n"
    f"Key findings:\n"
    f"- High memory usage on Tomcat01 and Tomcat03: 'MEMUsedMemPerc' shows 21/21 anomaly points with values ~97–98% (earliest 16:00). These are sustained high memory readings and are top concerns.\n"
    f"- Tomcat02 and Tomcat04 show process-related anomalies (zabbix agent process metric) across the window (18/18 points flagged).\n"
    f"- CPU-related issues observed on multiple Tomcats: single-CPU util metrics hit 100% in some cores (see `cpu_related_top10`), indicating CPU spikes on Tomcat01 and Tomcat02/03 in places.\n"
    f"- Network indicators on Tomcat03 show abnormalities (e.g., TCP-CLOSE-WAIT with 12/12 anomalies and elevated TotalTcpConnNum), and several hosts show bursts in packets/KB counts.\n\n"
    f"Interpretation / decision guidance:\n"
    f"- The anomalies are not isolated to a single downstream Tomcat. Multiple Tomcats (notably Tomcat01 and Tomcat03) show resource pressure (memory and some CPU/network issues).\n"
    f"- This pattern suggests broader resource contention or systemic issue rather than a single faulty downstream service. Start deeper investigation with priority on:\n"
    f"  1) JVM memory on Tomcat01 and Tomcat03 (heap/JVMUsedMemory, MEMUsedMemPerc) and GC behavior.\n"
    f"  2) CPU hotspots on the reported SingleCpu metrics for Tomcat01/02/03.\n"
    f"  3) Network connection accumulation on Tomcat03 (TCP-CLOSE-WAIT, TotalTcpConnNum) and packet/KB spikes.\n\n"
    f"Examples (top anomalies):\n  - " + "\n  - ".join(examples) + "\n\n"
    f"Stored outputs for follow-up: `metric_tomcat_anom_top20` (compact top table), `jvm_related_top10`, `cpu_related_top10`, `net_related_top10`."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  total_points_window  anomaly_count earliest_anomaly_timestamp  max_value_window  min_value_window  global_p95  global_p5
0   Tomcat01       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc                   21             21   2021-03-12 16:00:00+0800           98.0000           97.0000     98.0000    98.0000
1   Tomcat03       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc                   21             21   2021-03-12 16:00:00+0800           98.0000           97.0000     98.0000    97.0000
2   Tomcat02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...                   18             18   2021-03-12 16:01:00+0800            0.4000            0.3000      0.4000     0.3000
3   Tomcat04  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...                   18             18   2021-03-12 16:01:00+0800            0.4000            0.3000      0.4000     0.3000
4   Tomcat03     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT                   12             12   2021-03-12 16:02:00+0800            2.0000            0.0000      0.9000     0.0000
5   Tomcat03  OSLinux-OSLinux_FILE_-tmp-zabbix_agentd.log_Fi...                   18             11   2021-03-12 16:01:00+0800            0.3531            0.3524      0.3531     0.3524
6   Tomcat03            OSLinux-OSLinux_FILESYSTEM_-_FSCapacity                   11             11   2021-03-12 16:00:00+0800           30.5754           30.5753     30.5754    30.5753
7   Tomcat03        OSLinux-OSLinux_FILESYSTEM_-home_FSCapacity                   11             11   2021-03-12 16:00:00+0800            2.9886            2.9885      2.9886     2.9885
8   Tomcat03  OSLinux-OSLinux_FILESYSTEM_-tomcat_FSInodeUsed...                   11             11   2021-03-12 16:00:00+0800            0.0450            0.0449      0.0450     0.0449
9   Tomcat01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps                   10             10   2021-03-12 16:01:00+0800            1.0000            0.0000      1.0000     0.0000
10  Tomcat02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps                   10             10   2021-03-12 16:01:00+0800            1.0000            0.0000      0.5500     0.0000
11  Tomcat02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps                   10             10   2021-03-12 16:01:00+0800            1.0000            0.0000      1.0000     0.0000
12  Tomcat03     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps                   10             10   2021-03-12 16:00:00+0800            3.0000            0.0000      1.6500     0.0000
13  Tomcat03     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps                   10             10   2021-03-12 16:01:00+0800            1.0000            0.0000      1.0000     0.0000
14  Tomcat04  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...                   10             10   2021-03-12 16:00:00+0800            0.1000            0.0000      0.1000     0.0000
15  Tomcat04     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps                   10             10   2021-03-12 16:01:00+0800            1.0000            0.0000      0.5500     0.0000
16  Tomcat04     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps                   10             10   2021-03-12 16:01:00+0800            1.0000            0.0000      1.0000     0.0000
17  Tomcat01  OSLinux-OSLinux_FILE_-tmp-zabbix_agentd.log_Fi...                   17              9   2021-03-12 16:00:00+0800            0.3421            0.3414      0.3421     0.3414
18  Tomcat01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps                   10              9   2021-03-12 16:01:00+0800           12.0000            0.0000      7.0500     0.0000
19  Tomcat03  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...                   10              9   2021-03-12 16:00:00+0800            0.3000            0.0000      0.3000     0.0000,     cmdb_id                                        kpi_name  total_points_window  anomaly_count  max_value_window  min_value_window    global_p95     global_p5 earliest_anomaly_timestamp
0  Tomcat01         Tomcat-MEMORY_7441-MEMORY_JVMUsedMemory                    6              2      2.206537e+09      1.389012e+09  2.184340e+09  1.398842e+09   2021-03-12 16:01:00+0800
1  Tomcat02         Tomcat-MEMORY_7441-MEMORY_JVMUsedMemory                    6              2      1.537867e+09      9.061713e+08  1.512685e+09  9.546625e+08   2021-03-12 16:16:00+0800
2  Tomcat03         Tomcat-MEMORY_7441-MEMORY_JVMUsedMemory                    6              2      1.675509e+09      1.203301e+09  1.654786e+09  1.234393e+09   2021-03-12 16:01:00+0800
3  Tomcat04         Tomcat-MEMORY_7441-MEMORY_JVMUsedMemory                    6              2      1.723303e+09      1.249368e+09  1.710474e+09  1.255754e+09   2021-03-12 16:01:00+0800
4  Tomcat03         Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory                    5              2      2.984318e+09      2.512110e+09  2.959445e+09  2.528689e+09   2021-03-12 16:04:00+0800
5  Tomcat03  Tomcat-MEMORY_7441-MEMORY_JVMMemoryUsedPercent                    5              2      4.000000e+01      2.800000e+01  3.960000e+01  2.860000e+01   2021-03-12 16:04:00+0800
6  Tomcat01         Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory                    4              2      2.798608e+09      2.069871e+09  2.717866e+09  2.076874e+09   2021-03-12 16:07:00+0800
7  Tomcat01  Tomcat-MEMORY_7441-MEMORY_JVMMemoryUsedPercent                    4              2      5.000000e+01      3.300000e+01  4.985000e+01  3.495000e+01   2021-03-12 16:07:00+0800
8  Tomcat02         Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory                    4              2      3.281448e+09      2.649753e+09  3.252353e+09  2.702005e+09   2021-03-12 16:18:00+0800
9  Tomcat02  Tomcat-MEMORY_7441-MEMORY_JVMMemoryUsedPercent                    4              2      3.600000e+01      2.100000e+01  3.480000e+01  2.175000e+01   2021-03-12 16:18:00+0800,     cmdb_id                                           kpi_name  total_points_window  anomaly_count  max_value_window  min_value_window  global_p95  global_p5 earliest_anomaly_timestamp
0  Tomcat02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...                   18             18            0.4000            0.3000     0.40000    0.30000   2021-03-12 16:01:00+0800
1  Tomcat04  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...                   18             18            0.4000            0.3000     0.40000    0.30000   2021-03-12 16:01:00+0800
2  Tomcat02                    OSLinux-CPU_CPU-0_SingleCpuidle                    9              5           96.7849            0.0000    96.74626    0.00000   2021-03-12 16:02:00+0800
3  Tomcat02                    OSLinux-CPU_CPU-0_SingleCpuUtil                    8              5          100.0000            3.3117   100.00000   35.96656   2021-03-12 16:05:00+0800
4  Tomcat01                    OSLinux-CPU_CPU-0_SingleCpuUtil                   11              4          100.0000            2.3790   100.00000    2.64590   2021-03-12 16:13:00+0800
5  Tomcat03                    OSLinux-CPU_CPU-2_SingleCpuidle                   11              4           97.4359            0.0000    97.09565    0.00000   2021-03-12 16:07:00+0800
6  Tomcat04                    OSLinux-CPU_CPU-1_SingleCpuUtil                    9              4          100.0000           66.6002   100.00000   69.41840   2021-03-12 16:01:00+0800
7  Tomcat03                    OSLinux-CPU_CPU-2_SingleCpuUtil                    8              4          100.0000            2.5641   100.00000   33.17713   2021-03-12 16:07:00+0800
8  Tomcat04                            OSLinux-CPU_CPU_CPULoad                   18              3            1.4400            1.0000     1.35500    1.00000   2021-03-12 16:15:00+0800
9  Tomcat03                    OSLinux-CPU_CPU-1_SingleCpuidle                   11              3           98.1864            0.0000    97.98655    0.00000   2021-03-12 16:00:00+0800,     cmdb_id                                         kpi_name  total_points_window  anomaly_count  max_value_window  min_value_window   global_p95     global_p5 earliest_anomaly_timestamp
0  Tomcat03   OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT                   12             12            2.0000             0.000       0.9000       0.00000   2021-03-12 16:02:00+0800
1  Tomcat03  OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum                   21              4          987.0000           687.000     961.0000     714.00000   2021-03-12 16:24:00+0800
2  Tomcat04  OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum                   20              2          947.0000           727.000     939.4000     736.50000   2021-03-12 16:25:00+0800
3  Tomcat03  OSLinux-OSLinux_NETWORK_ens160_NETBandwidthUtil                   11              2            0.0073             0.002       0.0072       0.00225   2021-03-12 16:23:00+0800
4  Tomcat01  OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec                   10              2       596990.0000        189265.000  587641.7000  262999.75000   2021-03-12 16:13:00+0800
5  Tomcat01      OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn                   10              2          999.0000           746.000     982.8000     806.30000   2021-03-12 16:23:00+0800
6  Tomcat01     OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut                   10              2         1078.0000           811.000    1059.5500     874.00000   2021-03-12 16:23:00+0800
7  Tomcat02      OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn                   10              2         1057.0000           732.000    1012.0000     782.85000   2021-03-12 16:04:00+0800
8  Tomcat02     OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut                   10              2         1130.0000           794.000    1083.6500     845.75000   2021-03-12 16:04:00+0800
9  Tomcat04  OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec                   10              2       574077.0000        166982.000  568372.3500  195372.05000   2021-03-12 16:04:00+0800)```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz
import numpy as np

# Reuse existing df_metric and ensure it's prepared
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_metric = df_metric.dropna(subset=['value'])

# ensure timestamp_dt tz-aware in UTC+8
tz = pytz.timezone('Asia/Shanghai')
if 'timestamp_dt' not in df_metric.columns:
    df_metric['timestamp_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    tmp = pd.to_datetime(df_metric['timestamp_dt'], errors='coerce')
    if tmp.dt.tz is None:
        df_metric['timestamp_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        df_metric['timestamp_dt'] = tmp.dt.tz_convert(tz)

# parameters
components = ['IG01', 'IG02', 'Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04']
start = tz.localize(pd.to_datetime("2021-03-12 16:00:00"))
end = tz.localize(pd.to_datetime("2021-03-12 16:30:00"))

# 1) global thresholds (p95/p5) computed on full series (no time filter) for requested components
global_thresh_components = (
    df_metric[df_metric['cmdb_id'].isin(components)]
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(global_p95=lambda x: x.quantile(0.95), global_p5=lambda x: x.quantile(0.05))
    .reset_index()
)

# 2) filter to incident window and components
df_window_comp = df_metric[
    (df_metric['cmdb_id'].isin(components)) &
    (df_metric['timestamp_dt'] >= start) &
    (df_metric['timestamp_dt'] <= end)
].copy()

# merge thresholds into windowed data
merged_comp = df_window_comp.merge(global_thresh_components, on=['cmdb_id', 'kpi_name'], how='left')

# mark anomaly types
merged_comp['is_high'] = merged_comp['value'] >= merged_comp['global_p95']
merged_comp['is_low'] = merged_comp['value'] <= merged_comp['global_p5']
merged_comp['is_anomaly'] = merged_comp['is_high'] | merged_comp['is_low']

# aggregate per group in-window
agg_comp = (
    merged_comp
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        total_points_window=('value', 'count'),
        anomaly_count=('is_anomaly', 'sum'),
        count_high=('is_high', 'sum'),
        count_low=('is_low', 'sum'),
        max_value_window=('value', 'max'),
        min_value_window=('value', 'min'),
        global_p95=('global_p95', 'first'),
        global_p5=('global_p5', 'first'),
    )
)

# earliest anomaly timestamp per group
earliest_anom = (
    merged_comp[merged_comp['is_anomaly']]
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['timestamp_dt']
    .min()
    .rename(columns={'timestamp_dt': 'earliest_anomaly_timestamp'})
)
earliest_anom['earliest_anomaly_timestamp'] = earliest_anom['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')

# combine
agg_comp = agg_comp.merge(earliest_anom, on=['cmdb_id', 'kpi_name'], how='left')

# 3) Filter to anomaly_count>0 AND non-constant series (max != min)
agg_comp_filtered = agg_comp[(agg_comp['anomaly_count'] > 0) & (agg_comp['max_value_window'] != agg_comp['min_value_window'])].copy()

# If empty, preserved variable will be empty DataFrame
# 4) Compute direction and severity
def compute_direction_severity(row):
    gh = row['count_high'] > 0
    gl = row['count_low'] > 0
    max_v = row['max_value_window']
    min_v = row['min_value_window']
    gp95 = row['global_p95']
    gp5 = row['global_p5']
    # compute deviations
    high_dev = max_v - gp95 if not pd.isna(gp95) else np.nan
    low_dev = gp5 - min_v if not pd.isna(gp5) else np.nan
    # compute relative severities, handle zero denom
    def rel(dev, denom):
        if pd.isna(dev) or pd.isna(denom):
            return np.nan
        if denom == 0:
            return abs(dev)
        return abs(dev) / abs(denom)
    high_sev = rel(high_dev, gp95)
    low_sev = rel(low_dev, gp5)
    # choose direction
    if gh and not gl:
        return pd.Series(['high', high_sev if not pd.isna(high_sev) else abs(high_dev)])
    if gl and not gh:
        return pd.Series(['low', low_sev if not pd.isna(low_sev) else abs(low_dev)])
    if gh and gl:
        # choose larger extremal deviation (use severity as tie-break)
        if (high_sev if not pd.isna(high_sev) else 0) >= (low_sev if not pd.isna(low_sev) else 0):
            return pd.Series(['high', high_sev if not pd.isna(high_sev) else abs(high_dev)])
        else:
            return pd.Series(['low', low_sev if not pd.isna(low_sev) else abs(low_dev)])
    # fallback (shouldn't happen since anomaly_count>0)
    return pd.Series([None, np.nan])

if agg_comp_filtered.shape[0] > 0:
    agg_comp_filtered[['direction', 'severity']] = agg_comp_filtered.apply(compute_direction_severity, axis=1)
else:
    agg_comp_filtered['direction'] = pd.Series(dtype=str)
    agg_comp_filtered['severity'] = pd.Series(dtype=float)

# clean and round severity
agg_comp_filtered['severity'] = agg_comp_filtered['severity'].astype(float).round(6)

# 5) For each cmdb_id select KPI with maximum severity (tie-breaker: anomaly_count desc, earliest_anomaly_timestamp asc)
if agg_comp_filtered.shape[0] > 0:
    # convert earliest timestamp to datetime for tie-breaking (NaT if missing)
    agg_comp_filtered['_earliest_dt'] = pd.to_datetime(agg_comp_filtered['earliest_anomaly_timestamp'], format='%Y-%m-%d %H:%M:%S%z', errors='coerce')
    agg_comp_filtered = agg_comp_filtered.sort_values(['cmdb_id', 'severity', 'anomaly_count', '_earliest_dt'], ascending=[True, False, False, True])
    # pick top per cmdb_id
    idx = agg_comp_filtered.groupby('cmdb_id')['severity'].idxmax()
    top_per_cmdb = agg_comp_filtered.loc[idx].reset_index(drop=True)
    # ensure tie-breaker handled: if multiple with same severity, use sorted order to keep first
    # select and rename columns to required output
    top_severity_per_cmdb = top_per_cmdb[[
        'cmdb_id', 'kpi_name', 'direction', 'severity', 'earliest_anomaly_timestamp',
        'global_p95', 'global_p5', 'max_value_window', 'min_value_window', 'anomaly_count', 'total_points_window'
    ]].rename(columns={
        'kpi_name': 'top_kpi_name',
        'earliest_anomaly_timestamp': 'earliest_anomaly_timestamp_of_top_kpi',
        'anomaly_count': 'anomaly_count_of_top_kpi',
        'total_points_window': 'total_points_window_of_top_kpi'
    }).copy()
    # sort by severity desc and limit to top 6 rows (one per cmdb_id)
    top_severity_per_cmdb = top_severity_per_cmdb.sort_values('severity', ascending=False).head(6).reset_index(drop=True)
    # round numerical columns for compactness
    top_severity_per_cmdb[['global_p95', 'global_p5', 'max_value_window', 'min_value_window']] = top_severity_per_cmdb[['global_p95', 'global_p5', 'max_value_window', 'min_value_window']].round(6)
else:
    top_severity_per_cmdb = pd.DataFrame(columns=[
        'cmdb_id', 'top_kpi_name', 'direction', 'severity', 'earliest_anomaly_timestamp_of_top_kpi',
        'global_p95', 'global_p5', 'max_value_window', 'min_value_window', 'anomaly_count_of_top_kpi', 'total_points_window_of_top_kpi'
    ])

# 6) Components with no non-constant anomalies in-window
components_with_results = set(top_severity_per_cmdb['cmdb_id'].unique())
missing_components = [c for c in components if c not in components_with_results]

# compact note DataFrame for missing components
if len(missing_components) > 0:
    missing_components_df = pd.DataFrame({'cmdb_id': missing_components, 'note': ['No non-constant anomalies in-window']*len(missing_components)})
else:
    missing_components_df = pd.DataFrame(columns=['cmdb_id', 'note'])

# Final compact outputs (stored in kernel variables)
top_severity_per_cmdb, missing_components_df

```
Out[8]:
```
# Summarize the most severe anomaly per component (compact, plain English)
try:
    top = top_severity_per_cmdb
    missing = missing_components_df
except NameError:
    raise NameError("Expected variables `top_severity_per_cmdb` and `missing_components_df` not found in kernel.")

rows = []
for r in top.itertuples(index=False):
    rows.append(
        f"{r.cmdb_id}: top_kpi='{r.top_kpi_name}', direction={r.direction}, severity={r.severity:.6f}, "
        f"earliest={r.earliest_anomaly_timestamp_of_top_kpi}, global_p95={r.global_p95}, global_p5={r.global_p5}, "
        f"max={r.max_value_window}, min={r.min_value_window}, anomalies={int(r.anomaly_count_of_top_kpi)}/{int(r.total_points_window_of_top_kpi)}"
    )

if missing.shape[0] == 0:
    missing_note = "No component is missing non-constant anomalies in-window."
else:
    missing_note = "Components with no non-constant anomalies in-window: " + ", ".join(missing['cmdb_id'].tolist())

summary = (
    "Most severe anomaly per component (2021-03-12 16:00–16:30 UTC+8):\n"
    + "\n".join(rows)
    + "\n\n" + missing_note
    + "\n\nInterpretation hints:\n"
    "- IG01: CPU WIO spike (severity ~1.71) at 16:04 — indicates high I/O wait relative to its normal (p95).\n"
    "- IG02: sda disk bandwidth spike (severity 1.40) at 16:00 with many anomaly points — indicates heavy disk throughput.\n"
    "- Tomcat03: TCP CLOSE-WAIT accumulation (severity 1.22) at 16:02 — suggests network/connection handling problems.\n"
    "- Tomcat01: Thread count spike (severity 1.15) at 16:00 — possible thread surge or leak affecting service.\n"
    "- Tomcat04: CPU WIO elevated (severity 1.41) at 16:07 — I/O wait also notable on this host.\n"
    "- Tomcat02: Selected top KPI shows low-direction severity (~0.91) around 16:05 (single-CPU util low vs global_p5) — indicates notable drop below its historical low for that metric.\n"
    "\nRecommendation: prioritize investigation of disk I/O on IG02, I/O wait on IG01/Tomcat04, connection handling on Tomcat03, and JVM/threading on Tomcat01."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id                                       top_kpi_name direction  severity earliest_anomaly_timestamp_of_top_kpi  global_p95  global_p5  max_value_window  min_value_window  anomaly_count_of_top_kpi  total_points_window_of_top_kpi
0      IG01                             OSLinux-CPU_CPU_CPUWio      high  1.707420              2021-03-12 16:04:00+0800    1.170413   0.020800            3.1688            0.0166                         5                              31
1  Tomcat04                             OSLinux-CPU_CPU_CPUWio      high  1.406101              2021-03-12 16:07:00+0800    0.741490   0.031425            1.7841            0.0208                         2                              18
2      IG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps      high  1.400000              2021-03-12 16:00:00+0800    2.500000   0.000000            6.0000            0.0000                        13                              15
3  Tomcat03     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT      high  1.222222              2021-03-12 16:02:00+0800    0.900000   0.000000            2.0000            0.0000                        12                              12
4  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...      high  1.153846              2021-03-12 16:00:00+0800   13.000000   0.000000           28.0000            0.0000                         6                              31
5  Tomcat02                    OSLinux-CPU_CPU-0_SingleCpuUtil       low  0.907923              2021-03-12 16:05:00+0800  100.000000  35.966560          100.0000            3.3117                         5                               8, Empty DataFrame
Columns: [cmdb_id, note]
Index: [])```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz
import re

# Reuse df_trace if present, otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")

# Ensure numeric values and drop NaNs
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
df_trace = df_trace.dropna(subset=['value'])

# Ensure timezone-aware timestamp in UTC+8 (Asia/Shanghai)
tz = pytz.timezone('Asia/Shanghai')
if 'timestamp_dt' not in df_trace.columns:
    df_trace['timestamp_dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    tmp = pd.to_datetime(df_trace['timestamp_dt'], errors='coerce')
    if tmp.dt.tz is None:
        df_trace['timestamp_dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        df_trace['timestamp_dt'] = tmp.dt.tz_convert(tz)

# 1) Compute global p95 per trace_name using full series (no time filter)
trace_global_p95 = (
    df_trace
    .groupby('trace_name', as_index=False)['value']
    .agg(global_p95=lambda x: x.quantile(0.95))
)

# 2) Filter to incident window and source cmdb_id in [IG01, IG02]
start = tz.localize(pd.to_datetime("2021-03-12 16:00:00"))
end = tz.localize(pd.to_datetime("2021-03-12 16:30:00"))
df_trace_window = df_trace[
    (df_trace['timestamp_dt'] >= start) &
    (df_trace['timestamp_dt'] <= end) &
    (df_trace['cmdb_id'].isin(['IG01', 'IG02']))
].copy()

# 3) Merge global p95 into windowed data and compute required aggregates
merged = df_trace_window.merge(trace_global_p95, on='trace_name', how='left')
merged['is_anomaly'] = merged['value'] >= merged['global_p95']

# per (cmdb_id, trace_name) aggregates
agg = (
    merged
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(
        count_in_window=('value', 'count'),
        earliest_timestamp=('timestamp_dt', 'min'),
        mean_value=('value', 'mean'),
        max_value=('value', 'max'),
        anomaly_count=('is_anomaly', 'sum')
    )
)

# earliest anomaly timestamp per group (if any)
earliest_anom = (
    merged[merged['is_anomaly']]
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['timestamp_dt']
    .min()
    .rename(columns={'timestamp_dt': 'earliest_anomaly_timestamp'})
)

# combine
agg = agg.merge(earliest_anom, on=['cmdb_id', 'trace_name'], how='left')

# format timestamps and numeric columns compactly
agg['earliest_timestamp'] = agg['earliest_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
agg['earliest_anomaly_timestamp'] = agg['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
for c in ['mean_value', 'max_value']:
    agg[c] = agg[c].round(6)

# bring in global_p95 for display (already computed per trace_name)
agg = agg.merge(trace_global_p95, on='trace_name', how='left')
agg['global_p95'] = agg['global_p95'].round(6)

# 4) Return compact table limited to top 50 rows sorted by anomaly_count desc
trace_window_agg_top50 = agg.sort_values(['anomaly_count', 'count_in_window'], ascending=[False, False]).head(50).reset_index(drop=True)

# 5) Additional summary: trace.to_TomcatXX.duration_p95 or duration_mean groups
pattern = r'^trace\.to_Tomcat\d+\.duration_(p95|mean)$'
mask_tomcat_traces = trace_window_agg_top50['trace_name'].str.contains(pattern, case=True, na=False, regex=True)
tomcat_trace_summary_top20 = (
    trace_window_agg_top50[mask_tomcat_traces]
    .loc[:, ['cmdb_id', 'trace_name', 'anomaly_count', 'earliest_anomaly_timestamp']]
    .sort_values(['anomaly_count', 'earliest_anomaly_timestamp'], ascending=[False, True])
    .head(20)
    .reset_index(drop=True)
)

# Store full intermediate results in kernel variables (already done: trace_global_p95, merged, agg)
# Final compact outputs (select DataFrames to display)
trace_window_agg_top50, tomcat_trace_summary_top20, trace_global_p95.head(20)

```
Out[9]:
```
# Summarize trace analysis conclusions in plain English using existing kernel variables
try:
    trace_window_agg_top50
    tomcat_trace_summary_top20
    trace_global_p95
except NameError:
    raise NameError("Expected variables (trace_window_agg_top50, tomcat_trace_summary_top20, trace_global_p95) not found in kernel.")

summary = (
    "Trace analysis summary (2021-03-12 16:00–16:30 UTC+8) for IG01/IG02:\n\n"
    "Key observations:\n"
    "- IG nodes (IG01, IG02) show the largest and earliest trace anomalies within the window:\n"
    "  * IG01: trace.self.duration_p95 had 6 anomaly points (earliest 2021-03-12 16:03:00+0800), mean ~2.98s, max ~10.18s, global p95 ~3.0987s — indicates repeated high-tail latency spikes around ~16:03–16:04.\n"
    "  * IG01 and IG02 trace.self.duration_mean/p95 groups also have multiple anomalies (IG02 duration_mean: 5 anomalies, earliest 16:04; IG02 duration_p95: 4 anomalies, earliest 16:04), showing elevated latencies at the IG layer.\n\n"
    "- Traces to downstream Tomcats (trace.to_TomcatXX.duration_mean / duration_p95):\n"
    "  * Several 'to_Tomcat' groups show 1–2 anomaly points each, with earliest anomaly timestamps clustered later in the window (~16:25–16:26). See `tomcat_trace_summary_top20` for the list (examples: trace.to_Tomcat01.duration_mean, trace.to_Tomcat02.duration_mean, trace.to_Tomcat02.duration_p95, etc.).\n"
    "  * These downstream anomalies are sparse (low counts per trace group) and occur later than the concentrated IG anomalies.\n\n"
    "Interpretation / likely root cause:\n"
    "- The heavier and earlier anomaly signal is at the IG nodes (IG01/IG02). Multiple IG-level duration metrics exceed their global p95 concurrently and earlier in the window, which points to the IG layer as the primary source of latency.\n"
    "- Downstream Tomcats show a few late, low-count latency anomalies, likely as secondary/backpressure effects rather than the primary origin.\n\n"
    "Recommended next steps:\n"
    "1) Investigate IG01/IG02 around 16:03–16:04: check request queuing, resource contention, and any spike causes (GC, thread pools, network stalls) at IG.\n"
    "2) After stabilizing IG, validate whether the sparse Tomcat anomalies persist; if they do, investigate the specific Tomcat instances listed in `tomcat_trace_summary_top20`.\n\n"
    "Stored compact outputs for follow-up:\n"
    "- Detailed per-(cmdb_id,trace_name) window table: `trace_window_agg_top50` (top groups sorted by anomaly_count).\n"
    "- Downstream-Tomcat-specific summary: `tomcat_trace_summary_top20` (trace.to_TomcatXX.duration_* groups with anomaly counts and earliest anomaly times).\n"
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                      trace_name  count_in_window        earliest_timestamp   mean_value    max_value  anomaly_count earliest_anomaly_timestamp   global_p95
0     IG01         trace.self.duration_p95               30  2021-03-12 16:00:00+0800     2.981140    10.179150              6   2021-03-12 16:03:00+0800     3.098692
1     IG02        trace.self.duration_mean               30  2021-03-12 16:00:00+0800     0.671753     3.042090              5   2021-03-12 16:04:00+0800     0.666853
2     IG01        trace.self.duration_mean               30  2021-03-12 16:00:00+0800     0.681692     3.139906              4   2021-03-12 16:04:00+0800     0.666853
3     IG02         trace.self.duration_p95               30  2021-03-12 16:00:00+0800     2.933000    10.199000              4   2021-03-12 16:04:00+0800     3.098692
4     IG01     trace.to_Tomcat04.row_count               30  2021-03-12 16:00:00+0800   219.800000   259.000000              3   2021-03-12 16:07:00+0800   255.050000
..     ...                             ...              ...                       ...          ...          ...            ...                        ...          ...
25    IG02     trace.to_Tomcat02.row_count               30  2021-03-12 16:00:00+0800   215.200000   275.000000              1   2021-03-12 16:02:00+0800   246.150000
26    IG02  trace.to_Tomcat04.duration_p95               30  2021-03-12 16:00:00+0800     3.015008    10.021000              1   2021-03-12 16:25:00+0800    10.016665
27    IG01            trace.self.row_count               30  2021-03-12 16:00:00+0800  1718.866667  1988.000000              0                        NaN  3298.650000
28    IG02            trace.self.row_count               30  2021-03-12 16:00:00+0800  1716.600000  2188.000000              0                        NaN  3298.650000
29    IG02     trace.to_Tomcat04.row_count               30  2021-03-12 16:00:00+0800   212.333333   248.000000              0                        NaN   255.050000

[30 rows x 9 columns],    cmdb_id                       trace_name  anomaly_count earliest_anomaly_timestamp
0     IG01  trace.to_Tomcat01.duration_mean              2   2021-03-12 16:25:00+0800
1     IG01  trace.to_Tomcat02.duration_mean              2   2021-03-12 16:25:00+0800
2     IG01   trace.to_Tomcat02.duration_p95              2   2021-03-12 16:25:00+0800
3     IG01   trace.to_Tomcat04.duration_p95              2   2021-03-12 16:25:00+0800
4     IG02   trace.to_Tomcat01.duration_p95              2   2021-03-12 16:25:00+0800
5     IG02  trace.to_Tomcat03.duration_mean              2   2021-03-12 16:25:00+0800
6     IG02   trace.to_Tomcat03.duration_p95              2   2021-03-12 16:25:00+0800
7     IG02  trace.to_Tomcat04.duration_mean              2   2021-03-12 16:25:00+0800
8     IG01  trace.to_Tomcat03.duration_mean              1   2021-03-12 16:25:00+0800
9     IG01   trace.to_Tomcat03.duration_p95              1   2021-03-12 16:25:00+0800
10    IG01  trace.to_Tomcat04.duration_mean              1   2021-03-12 16:25:00+0800
11    IG02  trace.to_Tomcat01.duration_mean              1   2021-03-12 16:25:00+0800
12    IG02   trace.to_Tomcat04.duration_p95              1   2021-03-12 16:25:00+0800
13    IG01   trace.to_Tomcat01.duration_p95              1   2021-03-12 16:26:00+0800
14    IG02  trace.to_Tomcat02.duration_mean              1   2021-03-12 16:26:00+0800
15    IG02   trace.to_Tomcat02.duration_p95              1   2021-03-12 16:26:00+0800,                            trace_name  global_p95
0       trace.from_IG01.duration_mean    2.524549
1        trace.from_IG01.duration_p95   10.016047
2           trace.from_IG01.row_count  255.050000
3       trace.from_IG02.duration_mean    2.394702
4        trace.from_IG02.duration_p95   10.016668
5           trace.from_IG02.row_count  247.050000
6       trace.from_MG01.duration_mean    2.050712
7        trace.from_MG01.duration_p95   11.960253
8           trace.from_MG01.row_count  232.050000
9       trace.from_MG02.duration_mean    2.102059
10       trace.from_MG02.duration_p95   15.489940
11          trace.from_MG02.row_count  228.050000
12  trace.from_Tomcat01.duration_mean    2.955931
13   trace.from_Tomcat01.duration_p95   18.794885
14      trace.from_Tomcat01.row_count  253.150000
15  trace.from_Tomcat02.duration_mean    3.468983
16   trace.from_Tomcat02.duration_p95   20.544107
17      trace.from_Tomcat02.row_count  250.300000
18  trace.from_Tomcat03.duration_mean    1.940121
19   trace.from_Tomcat03.duration_p95    8.832080)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```